In [1]:
from pdf_with_tables import (get_page_elements, check_and_fix_nested_tables,
                             change_table_format)

In [2]:
pdf_pages = get_page_elements('assets/CP_Perjalanan Dinas.pdf')

In [5]:
table_test = pdf_pages[-2]['tables'][0]['table_dct']
table_test

[['Gol.',
  'Uang Saku Dinas',
  None,
  None,
  'Uang Makan Dinas',
  None,
  None,
  None,
  None,
  'Kelas\nPenerbangan'],
 [None,
  'Dalam Negeri\n(dalam ribuan\nRupiah)',
  'Asia1),\nAfrika\n(dalamUSD)',
  'Non Asia&\nAfrika,\nSingapore,\nHong Kong,\ndan Jepang\n(dalam USD)',
  'Dalam Negeri (dalam ribuan Rupiah)',
  None,
  None,
  'Asia1), Afrika\n(dalam USD)',
  'Non Asia&\nAfrika,\nSingapore,\nHong Kong, dan\nJepang\n(dalam USD)',
  None],
 [None,
  None,
  None,
  None,
  'Makan Pagi',
  'Makan\nSiang',
  'Makan\nMalam',
  None,
  None,
  None],
 ['10', '-', None, None, 'Sesuai Tagihan', None, None, None, None, 'Business'],
 ['9', None, None, None, None, None, None, None, None, 'Economy2)'],
 ['8', None, None, None, None, None, None, None, None, 'Economy'],
 ['7', '90', '16', '32', None, None, None, None, None, None],
 ['6', '60', None, None, '35', '65', '50', '33', '70', None],
 ['5', '40', '12', '25', '30', '50', '40', None, None, None],
 ['4', None, None, None, None, None,

In [28]:
# --- Run the processing ---
generated_chunks = process_triple_header_table_to_records(table_test)
generated_chunks

[{'Gol.': '10',
  'Uang Saku Dinas - Dalam Negeri (dalam ribuan Rupiah)': '-',
  'Uang Saku Dinas - Asia1), Afrika (dalamUSD)': None,
  'Uang Saku Dinas - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD)': None,
  'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Pagi': 'Sesuai Tagihan',
  'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Siang': None,
  'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Malam': None,
  'Uang Makan Dinas - Asia1), Afrika (dalam USD)': None,
  'Uang Makan Dinas - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD)': None,
  'Kelas Penerbangan': 'Business'},
 {'Gol.': '9',
  'Uang Saku Dinas - Dalam Negeri (dalam ribuan Rupiah)': None,
  'Uang Saku Dinas - Asia1), Afrika (dalamUSD)': None,
  'Uang Saku Dinas - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD)': None,
  'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Pagi': None,
  'Uang Makan Dinas - Dalam Negeri (

In [24]:
def check_triple_header(table_data):
    """
    Analyzes a table's structure and returns its type.
    """
    if len(table_data) < 2:
        return False # or "too_small"

    header_row_2 = table_data[1]
    header_row_3 = table_data[2]

    # Heuristic for the complex table:
    # Does the third row have non-None values under None values in the second?
    # This is a strong sign of a merged, hierarchical header.

    for i, item in enumerate(header_row_2):
#         print('item:', item, "below:", header_row_3[i])
        if item is None and header_row_3[i] is not None and i>0:
#             print('   ^True')
            return True
    
    # Heuristic for a simple table:
    # The first row is full and the second row looks like data.
    if all(isinstance(h, str) for h in header_row_2):
        return False
        
    return False # Default fallback

In [25]:
check_triple_header(table_test)

item: None below: None
item: Dalam Negeri
(dalam ribuan
Rupiah) below: None
item: Asia1),
Afrika
(dalamUSD) below: None
item: Non Asia&
Afrika,
Singapore,
Hong Kong,
dan Jepang
(dalam USD) below: None
item: Dalam Negeri (dalam ribuan Rupiah) below: Makan Pagi
item: None below: Makan
Siang
   ^True


True

In [26]:
check_triple_header(pdf_pages[-1]['tables'][0]['table_dct'])

item: None below: 10
item: Sumatera below: Sesuai tagihan, Hotel Bintang 5 (lima)
item: Jawa & Bali below: None
item: Kalimantan
(selain Kaltim) below: None
item: Kaltim &
Sulawesi below: None
item: Papua below: None
item: Asia1), Afrika
(dalam USD) below: None
item: Non Asia& Afrika,
Singapore, Hongkong dan
Jepang (dalam USD) below: None


False

In [27]:
pdf_pages[-1]['tables'][0]['table_dct']

[['Gol.',
  'Dalam Negeri (dalam ribuan Rupiah)',
  None,
  None,
  None,
  None,
  'Luar Negeri (dalam USD)',
  None],
 [None,
  'Sumatera',
  'Jawa & Bali',
  'Kalimantan\n(selain Kaltim)',
  'Kaltim &\nSulawesi',
  'Papua',
  'Asia1), Afrika\n(dalam USD)',
  'Non Asia& Afrika,\nSingapore, Hongkong dan\nJepang (dalam USD)'],
 ['10',
  'Sesuai tagihan, Hotel Bintang 5 (lima)',
  None,
  None,
  None,
  None,
  None,
  None],
 ['9', '1.200', '1.500', '1.200', '1.200', '1.500', '300', '400'],
 ['8', '1.000', '1.350', '1.000', '1.000', '1.350', '250', '350'],
 ['7', '900', '1200', '900', '900', '1.200', '200', '250'],
 ['6', '800', '900', '750', '800', '900', '80', '150'],
 ['5', '650', '750', '650', '700', '800', None, None],
 ['4', None, None, None, None, None, None, None],
 ['3', '450', '600', '400', '500', '600', None, None],
 ['2', '350', '500', '350', '400', '500', None, None]]

In [15]:
check_double_header(pdf_pages[6]['tables'][0]['table_dct'])

False